# Updated Colab Probe Training Path

Run this after the feature-extraction notebook has written global and dense patch feature caches. It trains the global probes, dense depth probes, dense surface-normal probes, then aggregates results and regenerates figures.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/cv-project')
os.environ['CV_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ.setdefault('PYTHONPATH', str(PROJECT_ROOT))
%cd {PROJECT_ROOT}

!nvidia-smi

In [ ]:
import sys

!{sys.executable} -m pip install -q -r requirements.txt
!{sys.executable} -m pip install -q pyarrow matplotlib seaborn scikit-learn

In [ ]:
# Global frozen-feature linear probes.
# Uses cached CLS/global features from data/exp1_under12h/features.
!PYTHONPATH=. python scripts/train_all_exp1_probes.py \
  --config configs/exp1_under12h.yaml \
  --device cuda \
  --batch-size 256

In [ ]:
# Dense patch-depth probes over the dense subset.
# The config runs CLIP-B/16 and DINOv2-B at final/layer8, within-texture only.
!PYTHONPATH=. python scripts/train_all_dense_depth_probes.py \
  --config configs/exp1_under12h_dense.yaml \
  --device cuda \
  --batch-size 256

In [ ]:
# Dense patch surface-normal probes over the same dense subset.
!PYTHONPATH=. python scripts/train_all_dense_surface_normal_probes.py \
  --config configs/exp1_under12h_dense.yaml \
  --device cuda \
  --batch-size 256

In [ ]:
# Aggregate and regenerate figures for both result directories.
!PYTHONPATH=. python scripts/aggregate_exp1_results.py --config configs/exp1_under12h.yaml
!PYTHONPATH=. python scripts/make_exp1_figures.py --config configs/exp1_under12h.yaml

!PYTHONPATH=. python scripts/aggregate_exp1_results.py --config configs/exp1_under12h_dense.yaml
!PYTHONPATH=. python scripts/make_exp1_figures.py --config configs/exp1_under12h_dense.yaml

In [ ]:
# Optional smoke checks after dense feature extraction.
RUN_SMOKE_CHECK = True
if RUN_SMOKE_CHECK:
    !PYTHONPATH=. python scripts/smoke_check_dense_depth.py \
      --render-manifest data/exp1_under12h_dense/manifests/render_valid_colab.parquet \
      --patch-cache data/exp1_under12h_dense/features/clip_vit_b16/final_patch.npz \
      --project-root . \
      --max-rows 16

# Experiment 1 Under-12h Probe Training

Run after feature caches have been extracted. These jobs train only linear/lightweight probes on frozen cached features.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/cv-project'
%cd $PROJECT_ROOT
%env CV_PROJECT_ROOT=$PROJECT_ROOT
!nvidia-smi

In [ ]:
!pip install -q -r requirements.txt
!pip install -q pyarrow

In [ ]:
# Global probes over all configured models/layers/tasks/textures.
!PYTHONPATH=. python scripts/train_all_exp1_probes.py \
  --config configs/exp1_under12h.yaml \
  --device cuda \
  --batch-size 256

In [ ]:
# Camera-distance add-on, if rendered/extracted.
!PYTHONPATH=. python scripts/train_all_exp1_probes.py \
  --config configs/exp1_under12h_camera_distance.yaml \
  --device cuda \
  --batch-size 256

In [ ]:
# Dense patch-depth sub-study. Within-texture only by default.
!PYTHONPATH=. python scripts/train_all_dense_depth_probes.py \
  --config configs/exp1_under12h_dense.yaml \
  --device cuda \
  --batch-size 256

In [ ]:
!PYTHONPATH=. python scripts/aggregate_exp1_results.py \
  --config configs/exp1_under12h.yaml \
  --output outputs/exp1_under12h/results/exp1_results_long.csv \
  --texture-drops-output outputs/exp1_under12h/results/exp1_texture_drops.csv

!PYTHONPATH=. python scripts/make_exp1_figures.py \
  --config configs/exp1_under12h.yaml \
  --results outputs/exp1_under12h/results/exp1_results_long.csv \
  --texture-drops outputs/exp1_under12h/results/exp1_texture_drops.csv \
  --output-dir outputs/exp1_under12h/figures